# Transcribe and translate the WhisperJAV audio

This free Colab notebook reads `RCT-915_part1.m4a` and `RCT-915_part2.m4a` from `My Drive/WhisperJAV`, transcribes the Japanese speech, and creates an English translation. It saves text and timestamped subtitle files back into that folder. The original audio parts are unchanged.

Choose a GPU runtime if available (the T4 you used before is suitable). No supporter license is needed. The large Whisper model may take a while to download and process long audio; Colab can disconnect, so the notebook checkpoints each completed part in Drive. If you want faster but potentially less accurate results, choose the `medium` model in the cell controls.

In [3]:
#@title Transcribe Japanese audio and translate it to English
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys, time, shutil

MODEL_SIZE = 'large-v3' #@param ['large-v3', 'medium']
ALSO_TRANSCRIBE_JAPANESE = True #@param {type:"boolean"}
LANGUAGE = 'ja'

drive.mount('/content/drive')
folder = Path('/content/drive/MyDrive/WhisperJAV')
part_paths = [folder / 'RCT-915_part1.m4a', folder / 'RCT-915_part2.m4a']
for p in part_paths:
    if not p.is_file():
        raise FileNotFoundError(f'Missing split audio part: {p}')
    if p.stat().st_size >= 100_000_000:
        raise RuntimeError(f'{p.name} is not under 100 MB; split it again first.')
    print(f'Found {p.name}: {p.stat().st_size / 1_000_000:.1f} MB')

if not shutil.which('ffprobe'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faster-whisper'], check=True)
import torch
from faster_whisper import WhisperModel
device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'
if device == 'cpu':
    print('No GPU detected. Transcription will work on CPU but may be very slow; a free Colab GPU is recommended.')
else:
    print('Using GPU:', torch.cuda.get_device_name(0))
print(f'Loading Whisper {MODEL_SIZE} ({compute_type}); the first model download can take several minutes...')
model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)

checkpoint = folder / 'RCT-915_whisper_progress.json'
if checkpoint.exists():
    saved = json.loads(checkpoint.read_text(encoding='utf-8'))
    if saved.get('model') != MODEL_SIZE or saved.get('language') != LANGUAGE:
        saved = {'model': MODEL_SIZE, 'language': LANGUAGE, 'parts': {}}
else:
    saved = {'model': MODEL_SIZE, 'language': LANGUAGE, 'parts': {}}
saved.setdefault('parts', {})
tasks = ['translate'] + (['transcribe'] if ALSO_TRANSCRIBE_JAPANESE else [])

for part_path in part_paths:
    part_name = part_path.name
    saved['parts'].setdefault(part_name, {})
    for task in tasks:
        if task in saved['parts'][part_name]:
            print(f'Skipping completed {part_name} / {task}; found it in the Drive checkpoint.')
            continue
        print(f'Starting {task} for {part_name}...')
        started = time.monotonic()
        segments, info = model.transcribe(
            str(part_path), language=LANGUAGE, task=task, beam_size=5, vad_filter=True,
        )
        rows = []
        for index, segment in enumerate(segments, start=1):
            rows.append({'start': float(segment.start), 'end': float(segment.end), 'text': segment.text.strip()})
            if index % 100 == 0:
                print(f'  {index} speech segments decoded...')
        saved['parts'][part_name][task] = rows
        checkpoint.write_text(json.dumps(saved, ensure_ascii=False), encoding='utf-8')
        print(f'Finished {part_name} / {task}: {len(rows)} segments in {(time.monotonic() - started) / 60:.1f} minutes.')

def probe_duration(path):
    result = subprocess.run([
        'ffprobe', '-v', 'error', '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1', str(path),
    ], text=True, capture_output=True, check=True)
    return float(result.stdout.strip())

def timestamp(seconds, srt=False):
    total_ms = max(0, round(seconds * 1000))
    hours, remainder = divmod(total_ms, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    secs, millis = divmod(remainder, 1000)
    separator = ',' if srt else '.'
    return f'{hours:02d}:{minutes:02d}:{secs:02d}{separator}{millis:03d}'

def save_task(task, txt_path, srt_path):
    all_rows = []
    offset = 0.0
    for part_path in part_paths:
        rows = saved['parts'].get(part_path.name, {}).get(task)
        if rows is None:
            return
        for row in rows:
            all_rows.append((row['start'] + offset, row['end'] + offset, row['text']))
        offset += probe_duration(part_path)
    txt_path.write_text('\n'.join(f'[{timestamp(start)}] {text}' for start, end, text in all_rows) + '\n', encoding='utf-8')
    with srt_path.open('w', encoding='utf-8') as f:
        for i, (start, end, text) in enumerate(all_rows, start=1):
            f.write(f'{i}\n{timestamp(start, True)} --> {timestamp(end, True)}\n{text}\n\n')
    print(f'Saved {txt_path.name} and {srt_path.name}')

save_task('translate', folder / 'RCT-915_english_translation.txt', folder / 'RCT-915_english_translation.srt')
if ALSO_TRANSCRIBE_JAPANESE:
    save_task('transcribe', folder / 'RCT-915_japanese_transcript.txt', folder / 'RCT-915_japanese_transcript.srt')
print('Done. The English translation and optional Japanese transcript are in WhisperJAV. The progress JSON can be kept to resume or deleted when no longer needed.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found RCT-915_part1.m4a: 91.7 MB
Found RCT-915_part2.m4a: 90.2 MB
No GPU detected. Transcription will work on CPU but may be very slow; a free Colab GPU is recommended.
Loading Whisper large-v3 (int8); the first model download can take several minutes...
Starting translate for RCT-915_part1.m4a...


TypeError: open() got an unexpected keyword argument 'metadata_errors'